# 05 — Preparação do Dataset: Somente Dados do Cliente

> **Dados do cliente são mantidos isolados** — nunca misturados com dados públicos.

Consome os segmentos rotulados de `data/client_labeled/` e gera um dataset YOLO  
completo em `data/datasets/client_only/`, pronto para `06_train_client_only.ipynb` ou `07_train_combined.ipynb`.

**Ordem típica:** após o `04` (idealmente `02` + `03` antes, para melhor pré-rotulagem).

**Classes:** `0=person` · `1=violence` · `2=pre_violence`

In [ ]:
!pip install -q ultralytics opencv-python-headless scikit-learn tqdm PyYAML matplotlib

In [ ]:
import cv2, yaml, json, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from IPython.display import display

LABELED_DIR  = Path("../data/client_labeled")        # saída do notebook 04
DATASET_DIR  = Path("../data/datasets/client_only")  # dataset YOLO isolado

EXTRACT_FPS  = 5
IMG_SIZE     = 640
PRE_V_THRESH = 0.45
CLASS_NAMES  = ["person", "violence", "pre_violence"]
# Em vídeos non_violence: se False, não atribui classe 2 só por pose (menos ruído no YOLO; recomendado).
ASSIGN_PRE_VIOLENCE_FROM_POSE_IN_NONVIOLENCE = False

for split in ["train","val","test"]:
    (DATASET_DIR/"images"/split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR/"labels"/split).mkdir(parents=True, exist_ok=True)
(DATASET_DIR/"metadata").mkdir(exist_ok=True)

assert LABELED_DIR.exists(), "Execute 04_client_classification.ipynb primeiro."

# Modelo de pose YOLOv8n-pose (substitui MediaPipe)
_pose_model = None
def _get_pose_model():
    global _pose_model
    if _pose_model is None:
        _pose_model = YOLO("yolov8n-pose.pt")
    return _pose_model

print(f"Origem  : {LABELED_DIR.resolve()}")
print(f"Dataset : {DATASET_DIR.resolve()}")

In [ ]:
# ── Detecção de pessoas (YOLOv8n COCO) ──────────────────────────────────────
_yolo_person = YOLO("yolov8n.pt")

def detect_bboxes(frame):
    res = _yolo_person(frame, classes=[0], verbose=False)[0]
    boxes = [tuple(int(v) for v in b) for b in res.boxes.xyxy.tolist()]
    return boxes or [(0, 0, frame.shape[1], frame.shape[0])]


# ── Score de postura agressiva via YOLOv8n-pose (sem MediaPipe) ──────────────
# Keypoints COCO: 0=nose 5=l_shoulder 6=r_shoulder 9=l_wrist 10=r_wrist 11=l_hip 12=r_hip
def pose_score(frame):
    pose_model = _get_pose_model()
    results = pose_model(frame, verbose=False, conf=0.3)
    best = 0.0
    for r in results:
        if r.keypoints is None or r.keypoints.xy is None:
            continue
        kps = r.keypoints.xy.cpu().numpy()  # [N, 17, 2]
        h, w = frame.shape[:2]
        for kp in kps:
            kpn = kp.astype(float)
            kpn[:, 0] /= max(w, 1)
            kpn[:, 1] /= max(h, 1)
            def pt(i): return kpn[i]
            try:
                sh_y = (pt(5)[1] + pt(6)[1]) / 2
                arm  = min(1.0, (max(0, sh_y - pt(9)[1]) +
                                  max(0, sh_y - pt(10)[1])) * 2)
                hip_c = (pt(11) + pt(12)) / 2
                vec   = pt(0) - hip_c
                tor   = min(1.0, abs(np.arctan2(vec[0], -vec[1])) / (np.pi / 3))
                spd   = min(1.0, np.linalg.norm(pt(9) - pt(10)) * 2)
                best  = max(best, float(np.clip(0.4*arm + 0.3*tor + 0.3*spd, 0, 1)))
            except Exception:
                continue
    return best


def to_yolo(w, h, bbox, cls):
    x1, y1, x2, y2 = bbox
    return f"{cls} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}"


def process_video(path, label):
    cap  = cv2.VideoCapture(str(path))
    if not cap.isOpened(): return []
    fps  = cap.get(cv2.CAP_PROP_FPS) or 25
    skip = max(1, int(fps / EXTRACT_FPS))
    recs, fi = [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if fi % skip == 0:
            h, w = frame.shape[:2]
            ps   = pose_score(frame)
            if label == "violence":
                cls = 1
            elif label == "non_violence" and ASSIGN_PRE_VIOLENCE_FROM_POSE_IN_NONVIOLENCE and ps >= PRE_V_THRESH:
                cls = 2
            else:
                cls = 0
            bboxes = detect_bboxes(frame)
            recs.append(dict(frame=frame, class_id=cls, pose_score=ps,
                             annots=[to_yolo(w, h, b, cls) for b in bboxes],
                             source=path.name))
        fi += 1
    cap.release()
    return recs

In [ ]:
# ── Split por VÍDEO (evita data leakage entre splits) ────────────────────────
# Frames do mesmo vídeo NUNCA caem em splits diferentes.
import random
random.seed(42)

video_splits = {}  # path -> split
for label in ["violence", "non_violence"]:
    vids = sorted(list((LABELED_DIR/label).glob("*.mp4")) +
                  list((LABELED_DIR/label).glob("*.avi")))
    if not vids: continue
    tr, tmp = train_test_split(vids, test_size=0.30, random_state=42)
    vl, te  = train_test_split(tmp,  test_size=0.33, random_state=42)
    for v in tr: video_splits[v] = "train"
    for v in vl: video_splits[v] = "val"
    for v in te: video_splits[v] = "test"

split_counts = {s: sum(1 for x in video_splits.values() if x==s) for s in ["train","val","test"]}
print("Vídeos por split:", split_counts)

# ── Processar e gravar frame a frame — SEM acumular na RAM ───────────────────
# Cada vídeo é processado, gravado no disco e descartado imediatamente.
# Isso evita OOM mesmo com centenas de vídeos.
split_counters  = {"train": 0, "val": 0, "test": 0}
class_counters  = {n: 0 for n in CLASS_NAMES}
total_frames    = 0

for label in ["violence", "non_violence"]:
    vids = sorted([v for v in video_splits if v.parent.name == label])
    for v in tqdm(vids, desc=label):
        split = video_splits[v]
        cap   = cv2.VideoCapture(str(v))
        if not cap.isOpened(): continue
        fps  = cap.get(cv2.CAP_PROP_FPS) or 25
        skip = max(1, int(fps / EXTRACT_FPS))
        fi   = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            if fi % skip == 0:
                h, w = frame.shape[:2]
                ps   = pose_score(frame)
                if label == "violence":
                    cls = 1
                elif label == "non_violence" and ASSIGN_PRE_VIOLENCE_FROM_POSE_IN_NONVIOLENCE and ps >= PRE_V_THRESH:
                    cls = 2
                else:
                    cls = 0
                bboxes = detect_bboxes(frame)
                annots = [to_yolo(w, h, b, cls) for b in bboxes]

                # Gravar imediatamente — não guardar na RAM
                k    = split_counters[split]
                name = f"{split}_{k:06d}"
                out_frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
                cv2.imwrite(str(DATASET_DIR/"images"/split/f"{name}.jpg"), out_frame)
                (DATASET_DIR/"labels"/split/f"{name}.txt").write_text("\n".join(annots))

                split_counters[split] += 1
                class_counters[CLASS_NAMES[cls]] += 1
                total_frames += 1
            fi += 1
        cap.release()

print(f"\nFrames totais : {total_frames}")
for n in CLASS_NAMES:
    print(f"  {n:14s}: {class_counters[n]}")

In [ ]:
# ── Frames já foram gravados no loop acima — aqui só YAML e stats ────────────
ds_yaml = dict(path=str(DATASET_DIR.resolve()),
               train="images/train", val="images/val", test="images/test",
               nc=3, names=CLASS_NAMES)
yaml_path = DATASET_DIR / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(ds_yaml, f, default_flow_style=False)

stats = dict(
    source="client_only",
    total=total_frames,
    split_by="video",
    videos_per_split=split_counts,
    frames_per_split=split_counters,
    classes=class_counters,
)
with open(DATASET_DIR/"metadata"/"stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print(f"\n✓ Dataset client_only pronto (split por vídeo — sem data leakage)")
print(f"  Vídeos  — train={split_counts['train']}  val={split_counts['val']}  test={split_counts['test']}")
print(f"  Frames  — train={split_counters['train']}  val={split_counters['val']}  test={split_counters['test']}")
print(f"  YAML: {yaml_path}")
print("\nPróximo: 06_train_client_only.ipynb (A) ou 07_train_combined.ipynb (B)")

In [ ]:
# Visualização
fig, axes = plt.subplots(1,2,figsize=(12,4))
colors=["steelblue","tomato","darkorange"]
axes[0].bar(CLASS_NAMES,[stats["classes"][n] for n in CLASS_NAMES],color=colors,edgecolor="black",alpha=0.85)
axes[0].set_title("Distribuição de Classes — client_only")
axes[1].bar(["train","val","test"],[stats["frames_per_split"][s] for s in ["train","val","test"]],
             color="slateblue",edgecolor="black",alpha=0.85)
axes[1].set_title("Frames por Split")
plt.suptitle("Dataset: Somente Dados do Cliente",fontsize=13)
plt.tight_layout()
plt.show()